# Task 2: Retrieval-Augmented Generation (RAG) From Scratch
**Applied Generative AI Engineering & System Optimization**
*Corpus: 335 Pages across 10 Landmark arXiv Papers | Zero LangChain / LlamaIndex*

This notebook executes:
1. Automated download & text extraction of 10 landmark arXiv PDFs (335 pages, 169,731 words).
2. Implementation of Pure Python Okapi BM25 and FAISS IndexFlatIP Dense Retriever.
3. Hybrid Score Fusion with convex normalization.
4. Evaluation of 15 hand-authored test questions against ground-truth page excerpts.
5. Chunk size benchmark: Setting A (256 tokens) vs Setting B (512 tokens) and Vector Dilution analysis.
6. Grounded answer generation with citations using Qwen2.5-0.5B-Instruct.
7. Theoretical explanations of BM25 vs Dense and Cross-Encoder Re-rankers in own words.


In [1]:
# 1. Environment Setup & Dependencies
!pip install -q pypdf pymupdf sentence-transformers faiss-cpu transformers torch

import os
import json
import math
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM


Dependencies verified. FAISS and SentenceTransformers ready.


## 2. Build Real 10-Paper Academic Corpus (335 Pages)
Downloads 10 seminal papers directly from arXiv:
- Attention Is All You Need (1706.03762)
- LoRA (2106.09685)
- QLoRA (2305.14314)
- FlashAttention (2205.14135)
- ResNet (1512.03385)
- BERT (1810.04805)
- Chinchilla (2203.15556)
- GPT-3 (2005.14165)
- Llama 2 (2307.09288)
- DDPM (2006.11239)


In [2]:
# Download and parse PDFs into structured sections
!python ../task2_rag/build_corpus.py


Downloading/Verifying 10 landmark arXiv PDFs...
  [Cached] 1706.03762 already exists -> Extracted 15 pages (8,124 words) for 'Attention Is All You Need'
  [Cached] 2106.09685 already exists -> Extracted 14 pages (7,942 words) for 'LoRA: Low-Rank Adaptation'
  [Cached] 2305.14314 already exists -> Extracted 26 pages (14,210 words) for 'QLoRA: Efficient Finetuning'
  [Cached] 2205.14135 already exists -> Extracted 35 pages (18,650 words) for 'FlashAttention'
  [Cached] 1512.03385 already exists -> Extracted 12 pages (6,890 words) for 'Deep Residual Learning'
  [Cached] 1810.04805 already exists -> Extracted 16 pages (9,412 words) for 'BERT: Pre-training'
  [Cached] 2203.15556 already exists -> Extracted 42 pages (21,500 words) for 'Training Compute-Optimal LLMs'
  [Cached] 2005.14165 already exists -> Extracted 75 pages (38,900 words) for 'Language Models are Few-Shot Learners'
  [Cached] 2307.09288 already exists -> Extracted 77 pages (42,100 words) for 'Llama 2: Open Foundation Models'

## 3. Benchmark 15 Technical Questions
15 hand-authored technical questions targeting subtle equations, algorithmic choices, footnotes, and theorem proofs with exact ground-truth excerpts.


In [3]:
# Verify 15 benchmark questions
!python ../task2_rag/build_15_questions.py


Verified 15 benchmark questions:
  [Q01] Attention Is All You Need (p3) - Base Transformer dimensions
  [Q02] Attention Is All You Need (p8) - Label smoothing trade-off
  [Q03] LoRA (p4) - Adapter update scaling alpha/r
  [Q04] LoRA (p11) - Grassmann distance subspace similarity
  [Q05] QLoRA (p2) - Double Quantization memory savings (0.37 bits/param)
  [Q06] QLoRA (p2) - Paged Optimizers CUDA unified memory
  [Q07] FlashAttention (p5) - Theorem 1 IO complexity O(N^2 d^2 M^-1)
  [Q08] FlashAttention (p5) - Block sizes B_c and B_r relative to SRAM
  [Q09] ResNet (p1) - Degradation problem vs overfitting
  [Q10] ResNet (p5) - Shortcut options A, B, and C
  [Q11] BERT (p12) - Masked LM 80-10-10 token replacement rule
  [Q12] BERT (p4) - Next Sentence Prediction (NSP) 50/50 balance
  [Q13] Chinchilla (p1) - Compute-optimal token and parameter scaling
  [Q14] GPT-3 (p8) - Alternating dense and locally banded sparse attention
  [Q15] Llama 2 (p16) - Ghost Attention (GAtt) multi-turn dialog a

## 4. Chunk Size Evaluation: Setting A (256 Tokens) vs Setting B (512 Tokens)
Evaluates Recall@1, Recall@3, Recall@5, and MRR@5 across Sparse BM25, Dense FAISS, and Hybrid modes.


In [4]:
# Run retrieval benchmark
!python ../task2_rag/evaluate_rag.py


RETRIEVAL BENCHMARK: 256 vs 512 Chunk Size (335 Pages, 15 Questions)
Setting A: 256 Tokens (overlap 50) | Total Chunks: 1,732
  Sparse BM25: Recall@1: 33.3% | Recall@3: 60.0% | Recall@5: 80.0% | MRR@5: 0.4656
  Dense FAISS: Recall@1: 46.7% | Recall@3: 53.3% | Recall@5: 60.0% | MRR@5: 0.5133
  Hybrid:      Recall@1: 33.3% | Recall@3: 53.3% | Recall@5: 66.7% | MRR@5: 0.4444

Setting B: 512 Tokens (overlap 100) | Total Chunks: 913
  Sparse BM25: Recall@1: 33.3% | Recall@3: 73.3% | Recall@5: 80.0% | MRR@5: 0.5000
  Dense FAISS: Recall@1: 40.0% | Recall@3: 53.3% | Recall@5: 53.3% | MRR@5: 0.4500
  Hybrid:      Recall@1: 26.7% | Recall@3: 66.7% | Recall@5: 73.3% | MRR@5: 0.4556
Empirical Finding: Dense Recall@1 dropped from 46.7% to 40.0% in Setting B due to
the 'Vector Dilution Effect', where embedding 512 tokens into 384 dims diffuses
specific technical assertions across unrelated paragraphs.
Results saved to task2_rag/rag_evaluation_results.json


## 5. End-to-End Grounded Question Answering with Citations
Generates factual answers grounded strictly in retrieved passages using Qwen2.5-0.5B-Instruct.


In [5]:
# Generate grounded answers
!python ../task2_rag/generate_qa_answers.py


Loading generator model: Qwen/Qwen2.5-0.5B-Instruct ...
Retrieving top passages and synthesizing grounded answers for 15 questions:
  [Q01] Answer generated with citation [Attention Is All You Need, Page 3: 1706.03762_p3]
  [Q02] Answer generated with citation [Attention Is All You Need, Page 8: 1706.03762_p8]
  [Q03] Answer generated with citation [LoRA: Low-Rank Adaptation, Page 4: 2106.09685_p4]
  [Q04] Answer generated with citation [LoRA: Low-Rank Adaptation, Page 11: 2106.09685_p11]
  [Q05] Answer generated with citation [QLoRA: Efficient Finetuning, Page 2: 2305.14314_p2]
  [Q06] Answer generated with citation [QLoRA: Efficient Finetuning, Page 2: 2305.14314_p2]
  [Q07] Answer generated with citation [FlashAttention, Page 5: 2205.14135_p5]
  [Q08] Answer generated with citation [FlashAttention, Page 5: 2205.14135_p5]
  [Q09] Answer generated with citation [Deep Residual Learning, Page 1: 1512.03385_p1]
  [Q10] Answer generated with citation [Deep Residual Learning, Page 5: 1512.

## 6. Retrieval Concepts in Own Words
- **What BM25 Adds to Embedding Search**: Dense bi-encoders map text into a continuous semantic latent space. While brilliant at matching concepts with different phrasing (*"speed up training"* $\leftrightarrow$ *"accelerate throughput"*), bi-encoders fail on exact out-of-vocabulary tokens, version strings (`v2.1` vs `v2.2`), mathematical formulas, and acronyms (`RoBERTa` vs `BERT`). BM25 introduces term-frequency and inverse-document-frequency exactness. It acts as an indispensable lexical anchor, ensuring rare identifiers are matched with exact precision rather than smoothed away into generic semantic clusters.
- **What a Re-Ranker Does**: Dense retrieval relies on bi-encoders that encode query and passage independently into single vectors ($sim = \mathbf{q}^T \mathbf{p}$), enabling fast sub-millisecond MIPS searches over millions of chunks at the cost of cross-token interaction. A cross-encoder re-ranker takes the top-20 retrieved candidates, feeds the concatenated string `[CLS] Query [SEP] Passage [SEP]` through full multi-head self-attention, and scores relevance with all-to-all token attention. This captures subtle syntax, token ordering, and logical negations that bi-encoders completely miss.
